In [15]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

## PIPELINE

El Pipeline convierte age y gender en variables binarias (OneHot), normaliza todas las variables numéricas (StandardScaler) y entrena un modelo de clasificación automáticamente. 

In [16]:
X = pd.read_csv("../data/clean/X_clean.csv")
y = pd.read_csv("../data/clean/y.csv").squeeze()

añadimos categoricas

In [17]:

cat_cols = ["age", "gender", "merchant", "customer", "category"]
num_cols = X.drop(columns=cat_cols).columns.tolist()

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
models = {
    "RandomForest_base": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    
    "XGBoost_base": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    
    "LightGBM_base": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )
}

In [21]:
import time 

def train_base_models(models):
    results = {}

    for name, model in models.items():
        
        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", model)
        ])
        
        # inicio
        start = time.perf_counter()

        clf.fit(X_train, y_train)

        # fin
        end = time.perf_counter()
        train_time = end - start

        # Predicción
        start_pred = time.perf_counter()

        y_pred_proba = clf.predict_proba(X_test)[:, 1]

        end_pred = time.perf_counter()
        pred_time = end_pred - start_pred
        
        # metrica
        auc = roc_auc_score(y_test, y_pred_proba)
        results[name] = {
            "AUC": auc,
            "Train Time (s)": train_time,
            "Predict Time (s)": pred_time
        }
        
        print(f"{name} → AUC: {auc:.4f} | Train: {train_time:.2f}s | Predict: {pred_time:.4f}s")

In [22]:
train_base_models(models)

RandomForest → AUC: 0.9952 | Train: 52.68s | Predict: 0.2180s
XGBoost → AUC: 0.9975 | Train: 6.62s | Predict: 0.0908s
[LightGBM] [Info] Number of positive: 469954, number of negative: 469954
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.308949 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 72090
[LightGBM] [Info] Number of data points in the train set: 939908, number of used features: 4174
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/opt/miniconda3/envs/tfg_env/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM → AUC: 0.9980 | Train: 7.48s | Predict: 0.3229s


In [23]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="AUC", ascending=False)

print(results_df)

                   AUC  Train Time (s)  Predict Time (s)
LightGBM      0.997961        7.389346          0.333923
XGBoost       0.997510        6.679053          0.098710
RandomForest  0.995239       52.279186          0.219570


## AJUSTAR HIPERPARÁMETROS

Vamos a realizar un ajuste de hiperparametros para nuestros modelos, para ello establecemos una función que prueba mediante gridsearch los distintos hiperparametros para nuestros modelos base.

In [25]:
def tune_fraud_models_(X_train, y_train, X_test, y_test,
                                 preprocessor, cv_folds=5, n_jobs=-1):

    pr_auc_scorer = make_scorer(average_precision_score,
                                needs_proba=True,
                                response_method="predict_proba")
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos
    print(f"📊 Ratio desbalance (neg/pos): {scale_pos_weight:.1f}x")

    models = {
        "random_forest": {
            "estimator": RandomForestClassifier(random_state=42, n_jobs=n_jobs),
            "params": {
                "classifier__n_estimators":          [200, 500, 800, 1200],
                "classifier__max_depth":             [10, 20, 30, 40, None],
                "classifier__min_samples_split":     [2, 5, 10, 20],
                "classifier__min_samples_leaf":      [1, 2, 4, 8],
                "classifier__max_features":          ["sqrt", "log2", 0.3, 0.5],
                "classifier__max_samples":           [0.7, 0.8, 0.9, None],
                "classifier__class_weight":          ["balanced", "balanced_subsample", None],
                "classifier__criterion":             ["gini", "entropy"],
                "classifier__min_impurity_decrease": [0.0, 0.001, 0.01],
            }
        },
        "lightgbm": {
            "estimator": lgb.LGBMClassifier(random_state=42, verbose=-1,
                                             n_jobs=n_jobs, boosting_type="gbdt"),
            "params": {
                "classifier__n_estimators":       [200, 500, 800, 1200],
                "classifier__max_depth":          [4, 6, 8, 10, -1],
                "classifier__learning_rate":      [0.005, 0.01, 0.05, 0.1, 0.2],
                "classifier__num_leaves":         [31, 63, 127, 255],
                "classifier__min_child_samples":  [5, 10, 20, 50],
                "classifier__subsample":          [0.6, 0.7, 0.8, 1.0],
                "classifier__subsample_freq":     [0, 1, 5],
                "classifier__colsample_bytree":   [0.6, 0.7, 0.8, 1.0],
                "classifier__reg_alpha":          [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__reg_lambda":         [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__min_split_gain":     [0.0, 0.01, 0.1],
                "classifier__class_weight":       ["balanced", None],
                "classifier__scale_pos_weight":   [1, scale_pos_weight],
                "classifier__path_smooth":        [0, 0.1, 1],
            }
        },
        "xgboost": {
            "estimator": xgb.XGBClassifier(
                random_state=42, eval_metric="aucpr",
                verbosity=0, use_label_encoder=False,
                n_jobs=n_jobs, tree_method="hist"
            ),
            "params": {
                "classifier__n_estimators":      [200, 500, 800, 1200],
                "classifier__max_depth":         [3, 4, 6, 8, 10],
                "classifier__learning_rate":     [0.005, 0.01, 0.05, 0.1, 0.2],
                "classifier__subsample":         [0.6, 0.7, 0.8, 1.0],
                "classifier__colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
                "classifier__colsample_bylevel": [0.6, 0.8, 1.0],
                "classifier__colsample_bynode":  [0.6, 0.8, 1.0],
                "classifier__reg_alpha":         [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__reg_lambda":        [0.1, 0.5, 1.0, 5.0],
                "classifier__min_child_weight":  [1, 3, 5, 10, 20],
                "classifier__gamma":             [0, 0.1, 0.5, 1.0, 2.0],
                "classifier__max_delta_step":    [0, 1, 5],
                "classifier__scale_pos_weight":  [1, scale_pos_weight],
                "classifier__grow_policy":       ["depthwise", "lossguide"],
                "classifier__max_bin":           [256, 512],
            }
        },
    }

    smote_params = {
        "smote__k_neighbors":       [3, 5, 7, 10],
        "smote__sampling_strategy": [0.1, 0.25, 0.5, "minority", "auto"],
    }

    results = {}

    for model_name, config in models.items():
        print(f"\n{'='*60}")
        print(f"  {model_name.upper()}")
        print(f"{'='*60}")

        for use_smote in [True, False]:
            tag = f"{model_name}_{'con' if use_smote else 'sin'}_smote"
            print(f"\n  → {'Con' if use_smote else 'Sin'} SMOTE...")

            if use_smote:
                pipeline = ImbPipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("smote",        SMOTE(random_state=42)),
                    ("classifier",   config["estimator"])
                ])
                param_grid = {**config["params"], **smote_params}
            else:
                pipeline = SkPipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("classifier",   config["estimator"])
                ])
                param_grid = config["params"]

            grid_search = GridSearchCV(
                estimator  = pipeline,
                param_grid = param_grid,
                scoring    = pr_auc_scorer,
                cv         = cv,
                n_jobs     = n_jobs,
                verbose    = 2,
                refit      = True,
                error_score= 0.0
            )

            grid_search.fit(X_train, y_train)

            # ── Métricas en test ──────────────────────────────────
            best_pipeline = grid_search.best_estimator_
            y_prob = best_pipeline.predict_proba(X_test)[:, 1]
            y_pred = best_pipeline.predict(X_test)

            auc_test    = roc_auc_score(y_test, y_prob)
            pr_auc_test = average_precision_score(y_test, y_prob)
            f2_test     = fbeta_score(y_test, y_pred, beta=2)
            f1_test     = fbeta_score(y_test, y_pred, beta=1)

            # ── Entrada en results ──
            results[tag] = {
                #Columnas de tu DataFrame original
                "AUC":             auc_test,

                # Columnas nuevas
                "PR_AUC_test":     pr_auc_test,
                "PR_AUC_cv":       grid_search.best_score_,
                "F2_test":         f2_test,
                "F1_test":         f1_test,
                "SMOTE":           use_smote,
                "model":           model_name,
                "best_params":     grid_search.best_params_,
                "best_estimator":  best_pipeline,
            }

            print(f"\n  ✅ AUC test    : {auc_test:.4f}")
            print(f"  ✅ PR-AUC test : {pr_auc_test:.4f}")
            print(f"  ✅ F2 test     : {f2_test:.4f}")
            print(f"  📌 Best params : {grid_search.best_params_}")

    return results


In [ ]:
results = tune_fraud_models_(
    X_train, y_train, X_test, y_test, preprocessor
)

# ── Tu DataFrame original + columnas nuevas ─
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="AUC", ascending=False)

# Separar best_estimator para no ensuciar la tabla
estimators = results_df.pop("best_params")        # params como columna aparte
best_models = results_df.pop("best_estimator")     # estimadores en Series separada

# Redondear métricas numéricas
metric_cols = ["AUC", "PR_AUC_test", "PR_AUC_cv", "F2_test", "F1_test"]
results_df[metric_cols] = results_df[metric_cols].astype(float).round(4)

print("\n📊 TABLA FINAL DE RESULTADOS")
print(results_df[["model", "SMOTE"] + metric_cols].to_string())

# ── Acceder al mejor modelo -
best_tag   = results_df.index[0]
best_model = best_models[best_tag]
print(f"\n🏆 Mejor modelo : {best_tag}")
print(f"   Params       : {estimators[best_tag]}")
print(f"\n{classification_report(y_test, best_model.predict(X_test), target_names=['legítima', 'fraude'])}")